# Day 3 - Streaming the answer

**Big idea:** users don't feel *total* time, they feel *how long they stared at a blank screen*. Streaming shows the first words right away.

## 1. The restaurant analogy

- **No streaming:** the kitchen cooks all 5 courses, then brings everything at once after 50 minutes. You sit at an empty table the whole time.
- **Streaming:** the first course arrives in 2 minutes and the rest keep coming. Total time is still 50 minutes - but it *feels* fine.

That first-plate time is **TTFT (time to first token)**. It's the number users actually feel.

## 2. `async` / `await` in one breath

`await something` means "this will take a while - go do other work and come back when it's ready." An `async def` function that uses `yield` is an **async generator**: it hands out values one at a time, pausing in between. Perfect for tokens trickling in over the network.

(Notebooks let you use `await` directly in a cell.)

In [1]:
import asyncio
import time

async def fake_llm(words, delay=0.2):
    """Pretend model: produces one word every `delay` seconds."""
    for w in words:
        await asyncio.sleep(delay)
        yield w

ANSWER = "Streaming makes the answer feel instant".split()

## 3. See TTFT vs total time with your own eyes

In [2]:
async def streamed():
    start = time.perf_counter()
    first = None
    async for word in fake_llm(ANSWER):
        if first is None:
            first = time.perf_counter() - start
        print(word, end=" ", flush=True)
    total = time.perf_counter() - start
    print(f"\n  streamed: first word after {first:.2f}s, total {total:.2f}s")

async def buffered():
    start = time.perf_counter()
    words = [w async for w in fake_llm(ANSWER)]   # wait for everything first
    total = time.perf_counter() - start
    print(" ".join(words))
    print(f"  buffered: first word after {total:.2f}s, total {total:.2f}s")

await streamed()
await buffered()

Streaming 

makes

the 

answer 

feel 

instant 


  streamed: first word after 0.20s, total 1.21s


Streaming makes the answer feel instant
  buffered: first word after 1.21s, total 1.21s


Same total time. But streamed, the user sees something after ~0.2 s instead of ~1.2 s.

## 4. What SSE looks like on the wire

**Server-Sent Events** is just text over a normal HTTP response. Each message is a few `field: value` lines ending with a blank line. The browser reads them as they arrive.

In [3]:
def sse_frame(data, event="message"):
    return f"event: {event}\ndata: {data}\n\n"

for word in ["Hello", " there", " friend"]:
    print(repr(sse_frame(word)))

'event: message\ndata: Hello\n\n'
'event: message\ndata:  there\n\n'
'event: message\ndata:  friend\n\n'


The header `Content-Type: text/event-stream` tells the browser "keep this connection open and read it piece by piece." In FastAPI I used `EventSourceResponse` from `sse-starlette`, which does this framing for me.

## 5. Timeouts: never wait forever

If the model server silently stops responding, code with no timeout waits **forever** - holding a connection, memory, and a file descriptor. Pile up enough of those and the server can't open new connections at all.

In the real app, `httpx.Timeout(connect=5, read=30, ...)` is the fix. Same idea with plain asyncio:

In [4]:
async def hangs_forever():
    await asyncio.sleep(3600)

try:
    await asyncio.wait_for(hangs_forever(), timeout=0.5)
except TimeoutError:
    print("gave up after 0.5s -> send the user one error chunk instead of spinning forever")

gave up after 0.5s -> send the user one error chunk instead of spinning forever


## 6. When the user closes the tab: `CancelledError`

If nobody stops the generator, the model keeps generating for a user who already left - wasting GPU time (Ollama) or real money (paid APIs).

**What I found on Day 3:** `request.is_disconnected()` never fired, because `sse-starlette` already runs its own listener that grabs the "disconnect" message first (each message can only be read once). Instead, sse-starlette **cancels** my generator's task, which raises `asyncio.CancelledError` inside it. So: catch it, clean up, and **re-raise**.

In [5]:
async def sse_stream():
    try:
        async for word in fake_llm(["one", "two", "three", "four", "five"], delay=0.3):
            print("sent:", word)
    except asyncio.CancelledError:
        print("client left -> stop the upstream call, clean up")
        raise   # always re-raise so asyncio knows the cancel worked

task = asyncio.create_task(sse_stream())
await asyncio.sleep(0.75)   # the user reads two words, then closes the tab
task.cancel()
try:
    await task
except asyncio.CancelledError:
    print("task is cancelled - nothing keeps running")

sent: one


sent: two
client left -> stop the upstream call, clean up
task is cancelled - nothing keeps running


## 7. Swapping providers without touching the streaming code

Each provider is just "a thing that yields text." Put them in a dict and look one up by name. The streaming / timeout / disconnect wrapper never changes.

In [6]:
async def ollama_tokens(prompt):
    for w in ["local", "answer"]:
        yield w

async def gemini_tokens(prompt):
    for w in ["cloud", "answer"]:
        yield w

TOKEN_SOURCES = {"ollama": ollama_tokens, "gemini": gemini_tokens}

async def chat(prompt, provider="ollama"):
    source = TOKEN_SOURCES.get(provider)
    if source is None:
        return f"400: unknown provider {provider!r}"
    return [w async for w in source(prompt)]

print(await chat("hi", "ollama"))
print(await chat("hi", "gemini"))
print(await chat("hi", "openai"))

['local', 'answer']
['cloud', 'answer']
400: unknown provider 'openai'


## Recap

- Users feel **TTFT**, not total latency -> stream.
- SSE = plain text frames (`event:` / `data:` / blank line) over an open HTTP response.
- Always set timeouts, or one stuck upstream call leaks resources forever.
- On disconnect, sse-starlette cancels the task -> catch `CancelledError`, clean up, re-raise.
- Keep providers swappable behind one shared streaming wrapper.